# Lab 10: Shipping with Agents: Build and Ship Safely

You'll drive an AI agent to build and ship a real website, and route every change through GitHub Issues and
pull requests with yourself as the reviewer. You direct and **verify**; the agent does the execution.

The focus is *how a change becomes official* (built, reviewed, and deliberately shipped), not clever prompting.
Speed of building is not the goal. Trustworthy shipping is.

---

## Learning Outcomes

By the end of this lab, you should be able to:
1. Drive **Claude Code** to build and **deploy a live website**, and verify the pipeline it writes.
2. Write a focused **CLAUDE.md** and explain why it *shapes* behaviour without *guaranteeing* it.
3. Drive work through **GitHub Issues and pull requests**, reviewing each change before it ships.
4. Explain why an agent owns *execution* while a human owns *what counts as correct* and *whether to ship*.

## 0. Your environment and software requirements

You'll work the way teams do in 2026: inside **VS Code**, driving **Claude Code** (the agent) to do the
execution (scaffolding files, running git, writing config) while *you* direct and **verify**. You rarely type
boilerplate by hand; you describe intent and check the result.

**Primary environment: the cockpit for this whole lab**
- **VS Code** + **Claude Code**. Everything is driven from here.

**Accounts**
- GitHub account: https://github.com
- A **paid** Claude plan (Pro / Max / Team / Enterprise) or Anthropic Console billing (required for Claude Code).

**Supporting tools.** Claude Code runs most of these for you, but they must be installed and on your PATH:

| Tool | Min version | Why |
|---|---|---|
| Git | 2.30+ | version control |
| GitHub CLI (`gh`) | 2.0+ | create repos, issues, and PRs |
| Node.js (incl. npm) | 20+ | runs Playwright (the tests) |
| Python 3 | 3.8+ | serves the site locally |
| A modern browser | any | GitHub settings; viewing your live site |

> Use **public** repos throughout. It keeps GitHub Pages and cross-repo CI free and tokenless.
> **Windows:** run inside **WSL** (recommended) or **Git Bash** so `bash`/`python3` behave as written.

> **How this lab works.** You're in VS Code with Claude Code open. Three kinds of instruction appear below:
> - **Prompt** (quoted blocks): what you say to Claude Code; it writes the files and runs the commands.
> - **Terminal** (code cells): the few things *you* run in VS Code's integrated terminal.
> - **Browser**: GitHub settings you click yourself (e.g. branch protection).
>
> Your job is to **direct and verify**. Read what the agent produced and confirm it's right, rather than
> hand-typing it.

In [ ]:
# One-time setup, then verify your toolchain (VS Code integrated terminal)
gh auth login                                   # GitHub.com -> HTTPS -> browser login
git config --global user.name  "Your Name"
git config --global user.email "you@example.com"

git --version
node --version        # v20+
python3 --version     # Windows: may be 'python --version'
gh --version
claude --version      # Claude Code; if missing, see install cell below

### If anything is missing: install it

If a command above printed **`command not found`**, install the missing tool, then **reopen your terminal**
(refreshes PATH) and re-check.

**macOS** (via [Homebrew](https://brew.sh)):
```bash
brew install git node python gh
brew install --cask visual-studio-code
```

**Windows** (`winget` in PowerShell; then work inside **WSL** or **Git Bash**):
```powershell
winget install Git.Git OpenJS.NodeJS.LTS Python.Python.3.12 GitHub.cli Microsoft.VisualStudioCode
```

**Linux (Debian / Ubuntu):**
```bash
sudo apt update && sudo apt install -y git python3 gh
curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - && sudo apt install -y nodejs
sudo snap install code --classic
```

**Claude Code** native installer (no Node.js needed):
```bash
curl -fsSL https://claude.ai/install.sh | bash      # macOS / Linux / WSL
irm https://claude.ai/install.ps1 | iex             # Windows PowerShell
claude --version    # verify     |     claude doctor    # diagnose
```
First run opens your browser to sign in. Requires a paid plan. The free Claude.ai plan does not include Claude
Code. Docs: https://docs.claude.com/en/docs/claude-code/overview

## Step 1: Build your portfolio and ship it

This is a **real deliverable**: a personal portfolio you can put on job applications. You'll set up the project
and its `CLAUDE.md` **first**, so the agent has your conventions from the start, then drive Claude Code to build
a polished page (one that automatically lists your GitHub repositories) and deploy it live.

### Step 1.1: Open the cockpit

Make an empty working folder, open it in **VS Code**, and start **Claude Code** (`claude` in the integrated
terminal, or the Claude Code panel). Everything below is a **prompt** to Claude Code unless marked *Terminal*
or *Browser*.

### Step 1.2: Set up the repo and write the CLAUDE.md first

Set the agent's standing context *before* you set it to work, so the build follows your conventions from the
start. Claude Code reads `CLAUDE.md` at the start of a session; it's your project's standing instructions, kept
in version control so every session begins from the same context.

**Prompt:**
> Create a new **public** GitHub repo called `<NAME>-site`, clone it here, and add a `CLAUDE.md` with the
> content below, plus a `.gitignore` (node_modules, OS files) and a short `README.md`. Commit and push.

**CLAUDE.md (keep it short; every line should change how the agent behaves):**
```
# <NAME>-site: personal portfolio
Plain HTML/CSS plus a little vanilla JS. No frameworks, no build step.
Deploy: push to main, GitHub Actions publishes to GitHub Pages.

## Conventions
- Site files live in public/.
- Section ids: #hero #about #skills #projects #repos #contact.
- Keep it responsive; no horizontal scroll at 375px.
```

> 💡 **Important.** A `CLAUDE.md` *shapes* behaviour, it does not *enforce* it. Anthropic's docs call it
> context, not enforced configuration. Rule of thumb: if breaking a rule should block a merge, that rule belongs
> in CI or branch protection, not here. In this lab the gate you rely on is the pull-request review in Step 2.

### Step 1.3: Build the portfolio and deploy

Now set the agent to work. It already has your conventions from `CLAUDE.md`, so the prompt only needs the task.

**Prompt:**
> Following the conventions in `CLAUDE.md`, build a **polished, modern personal portfolio** in `public/` that I
> can use when applying for jobs. Professional, not dated: clean typography, generous spacing, fully responsive,
> subtle transitions. Sections, each with its `id`:
> - **`#hero`**: my name, a one-line professional tagline, and links (email, GitHub, LinkedIn).
> - **`#about`**: a short professional bio.
> - **`#skills`**: my key skills / areas of focus.
> - **`#projects`**: a few curated project highlights (title, one-line description, link).
> - **`#repos`**: *automatically list my public GitHub repositories* by fetching
>   `https://api.github.com/users/<USER>/repos?sort=updated&per_page=12` **client-side in vanilla JS**, each as a
>   card (name, description, primary language, star count, link). Show a graceful message if there are none or
>   the request fails.
> - **`#contact`**: email, GitHub, and LinkedIn links.
>
> Then add `.github/workflows/deploy.yml` that deploys `public/` to **GitHub Pages on push to `main`**. Commit
> and push.

**Read what it proposes before approving**, especially the workflow file and how it handles the repo fetch (the
unauthenticated GitHub API allows about 60 requests per hour per visitor, which is fine for one fetch on load).

### Step 1.4: Verify, then go live

Don't take the pipeline on trust. *Verifying it is the skill this course trains.* Open
`.github/workflows/deploy.yml` and confirm:
- it triggers on **push to `main`**,
- it uploads the **`public/`** folder (`upload-pages-artifact` path),
- it grants Pages permissions (`pages: write`, `id-token: write`).

*Browser:* enable Pages once. Go to **repo, then Settings, then Pages, then Source, and choose
“GitHub Actions.”** This turns on the free hosting so your workflow has somewhere to publish to.

✅ **Checkpoint:** the **Actions** tab shows a green run; your site is live at
`https://<USER>.github.io/<NAME>-site/`.

## Step 2: Drive the work through GitHub Issues

In 2026 you don't just chat tasks at an agent and hope. You file each piece of work as a **GitHub Issue**, the
agent works one issue on a branch and opens a **pull request**, and you **review and merge**. You get a record
of what was asked, a place to check the work before it ships, and a clean handoff between you and the agent.

You'll run that loop now on your single site repo, with yourself as the only gate. (The automated test gate
comes in Step 3.)

### Step 2.1: Require a pull request before merging

So a change can't land on `main` (and deploy) without review, turn on a basic gate.

*Browser:* in `<NAME>-site`, go to **Settings, then Rules, then Rulesets, then New branch ruleset:**
- Enforcement **Active**; Target **Include default branch** (`main`)
- Tick **Restrict deletions**, **Block force pushes**, and **Require a pull request before merging**

✅ **Checkpoint:** pushing directly to `main` is now rejected. Every change has to arrive as a PR you review.
(If you later add automated tests, you would add their check to this same gate.)

### Step 2.2: File an issue

Pick a real improvement to your portfolio and write it as an issue. *Terminal* (or use the **New issue** button
on GitHub):

```bash
gh issue create --title "Add a dark-mode toggle" \
  --body "Add a button in the nav that toggles light and dark. Keep the choice for the session."
```

Use whatever enhancement you actually want: a dark-mode toggle, a richer projects section, better mobile
spacing.

### Step 2.3: Let the builder work the issue

In the site repo's Claude Code session.

**Prompt:**
> Work issue #1. Create a branch, implement the change, commit, push, and open a pull request that closes the
> issue.

Then **review the PR yourself**: read the diff, open the preview, and check it does what the issue asked. When
you're happy, **merge it**. The merge to `main` triggers `deploy.yml` and the change goes live.

✅ **Checkpoint:** an issue became a reviewed PR, which became a live change, with you as the gate. That loop
(issue, branch, PR, review, merge, deploy) is the spine of professional agent-driven work.

## Appendix: reference files (for verifying the agent's output)

Students drive Claude Code to *generate* this; use it to check what it produced. It is not meant to be
hand-typed.

**`.github/workflows/deploy.yml` (site repo):**
```yaml
name: Deploy to GitHub Pages
on:
  push: { branches: [main] }
  workflow_dispatch:
permissions: { contents: read, pages: write, id-token: write }
concurrency: { group: pages, cancel-in-progress: true }
jobs:
  deploy:
    environment: { name: github-pages, url: "${{ steps.deployment.outputs.page_url }}" }
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/configure-pages@v5
      - uses: actions/upload-pages-artifact@v3
        with: { path: ./public }
      - id: deployment
        uses: actions/deploy-pages@v4
```

> **One-line summary.** The agent owns execution; you own what counts as correct and whether to ship. A
> `CLAUDE.md` makes the right thing the default; requiring a reviewed pull request is what keeps an unreviewed
> change from going live.